# Run the OBP pipeline from a notebook
Same functions as the command-line scripts. Finished runs are skipped unless `force=True`, so re-running a cell is safe.

In [ ]:
import os, sys
sys.path.insert(0, "../code")                      # make the pipeline importable
# os.environ["DEEPSEEK_API_KEY"] = "..."           # or set it in your shell; never commit it

import pandas as pd
import run_experiment, evaluate_final, summarise

CONFIG = "configs/smoke.yaml"                       # fake LLM, no key; switch to configs/eoh_baseline.yaml for real runs

## 1. Search
One row per finished run. Use `methods=[...]` and `seeds=[...]` to run a subset.

In [ ]:
idx = run_experiment.main(CONFIG)                   # e.g. methods=["eoh"], seeds=[0]
idx[["label", "seed", "n_evals", "n_llm_calls", "cost_usd", "best_train_excess", "stop_reason"]]

## 2. Held-out evaluation

In [ ]:
summ = evaluate_final.main(CONFIG)
summ.pivot(index="label", columns="set", values="mean_excess").mul(100).round(2)   # % over lower bound

## 3. Tables and figures

In [ ]:
from IPython.display import Image, display
figs = summarise.main(CONFIG)
for name in ["convergence_evals", "test_sets"]:
    display(Image(figs / f"{name}.png", width=650))

## 4. Dig into one run

In [ ]:
from obpevo import config as C
run = C.results_dir(C.load(CONFIG)) / "eoh" / "seed0"
evals = pd.read_csv(run / "evals.csv")                # every candidate: status, score, train excess
print((run / "best_program.py").read_text())
evals.status.value_counts()